# 🧠 EX53: Training Settings (การตั้งค่าการฝึกสอน)

| พารามิเตอร์ | ค่าเริ่มต้น | ผลกระทบ |
|------------|------------|---------|
| `epochs` | 100 | จำนวนรอบเทรน |
| `batch` | 16 | ตัวอย่างต่อ gradient step |
| `imgsz` | 640 | ความละเอียดภาพ (px) |
| `lr0` | 0.01 | อัตราการเรียนรู้เริ่มต้น |
| `optimizer` | `AdamW` | อัลกอริทึม gradient descent |
| `amp` | True | Mixed precision FP16 — ลด VRAM ครึ่งหนึ่ง |

### AdamW vs SGD
**SGD+momentum:** $v_{t+1} = \mu v_t - \eta \nabla L$, $\theta \mathrel{+}= v_{t+1}$

**AdamW:** ปรับ LR แต่ละพารามิเตอร์โดยอัตโนมัติ พร้อม weight decay แยก
AdamW ลู่เข้าเร็วกว่า; SGD+cosine มักให้ mAP สูงกว่าบนชุดข้อมูลขนาดใหญ่

## ⚠️ คำเตือน
ตรวจสอบ `data.yaml` ก่อน `model.train()` เสมอ

## 🔗 ลิงก์
- [[YOLO_Learning_Plan]] | [[learning_journal]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, torch
import pandas as pd
import matplotlib.pyplot as plt
from solution import train_custom_model
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] ใช้อุปกรณ์: {device}")

print("\n--- เริ่มการตรวจสอบ ---")
print("เทรน 3 epoch บน coco8 เพื่อตรวจสอบ loss output...")
results = train_custom_model(model_path="yolo11n.pt", data_yaml="coco8.yaml",
                              epochs=3, batch=8, imgsz=320, device=device)

results_csv = results.save_dir / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print(f"\nคอลัมน์: {list(df.columns)}")
    print(df.tail(3).to_string(index=False))
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    fig, axes = plt.subplots(1, len(loss_cols), figsize=(4*len(loss_cols), 4))
    if len(loss_cols)==1: axes=[axes]
    for ax,col in zip(axes, loss_cols):
        ax.plot(df["epoch"], df[col], marker="o", linewidth=2)
        ax.set_title(col); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(True, alpha=0.3)
    plt.suptitle("กราฟ Loss (3 epoch audit)")
    plt.tight_layout(); plt.show()
print("--- สิ้นสุดการตรวจสอบ ---")
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
